In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("Equipo-Morado/df_limpio.csv")

In [3]:
df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,6B4H6E,22,16000.0,10000,769,22.0,1,8.96,48.0,0.32,Escolar,Desempleado,Soltero,1.0,1.0,Vivienda,0.0,0,65.85
1,VKTUXP,33,38308.0,16474,699,70.0,3,14.07,36.0,0.20,Máster,Jornada completa,Casado,1.0,1.0,Vivienda,0.0,0,260.60
2,5P9OVL,51,51271.0,40770,812,309.0,1,12.66,12.0,0.36,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,1.0,0,221.02
3,YL0W8G,30,31923.0,36707,813,43.0,1,8.96,24.0,0.32,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,0.0,0,125.00
4,YP8WTV,28,27373.0,86981,765,60.0,2,10.07,48.0,0.55,Escolar,Jornada completa,Soltero,0.0,1.0,Vivienda,0.0,0,800.00


In [4]:
import pandas as pd 
#Para calcular los prestamos solo necesitamos las columnas ID, Monto_Inicial, Ratio_Interes y Duracion por lo que creamos un nuevo dataframe con estas columnas.
cols_necesarias = ["ID", "Monto_Inicial", "Ratio_Interes", "Duracion"]
prestamos = df[cols_necesarias].copy()
print(prestamos)

            ID  Monto_Inicial  Ratio_Interes  Duracion
0       6B4H6E          10000           8.96      48.0
1       VKTUXP          16474          14.07      36.0
2       5P9OVL          40770          12.66      12.0
3       YL0W8G          36707           8.96      24.0
4       YP8WTV          86981          10.07      48.0
...        ...            ...            ...       ...
101894  5PM51H           5681          12.11      48.0
101895  L0RUJT          40000          10.77      48.0
101896  8OJZU0          41795          10.41      36.0
101897  II8B24          21847           6.26      36.0
101898  A8KWSO          40000           6.44      24.0

[101899 rows x 4 columns]


In [5]:
import pandas as pd
filas = []
for _, p in prestamos.iterrows():
    capital = p["Monto_Inicial"]
    n = int(p["Duracion"])
    i = p["Ratio_Interes"] / 100 / 12                # tipo de interes mensual

    cuota_fr = capital * i / (1 - (1 + i) ** -n)     # frances: cuota constante
    amort_al = capital / n                           # aleman: capital constante

    saldo_fr = capital
    saldo_al = capital
    for mes in range(1, n + 1):
        # Sistema frances
        interes_fr = saldo_fr * i
        amort_fr = cuota_fr - interes_fr
        # Sistema aleman
        interes_al = saldo_al * i
        cuota_al = amort_al + interes_al

        filas.append({
            "ID": p["ID"], "Mes": mes,
            "Fr_Cuota": cuota_fr, "Fr_Intereses": interes_fr,
            "Fr_Capital_Amortizado": amort_fr, "Fr_Capital_Pendiente": saldo_fr - amort_fr,
            "Fr_Flujo_Caja": -amort_fr, "Al_Cuota": cuota_al, "Al_Intereses": interes_al,
            "Al_Capital_Amortizado": amort_al, "Al_Capital_Pendiente": saldo_al - amort_al,
            "Al_Flujo_Caja": -amort_al
        })
        saldo_fr -= amort_fr
        saldo_al -= amort_al

cuadros = pd.DataFrame(filas).round(2)
cuadros

,ID,Mes,Fr_Cuota,Fr_Intereses,Fr_Capital_Amortizado,Fr_Capital_Pendiente,Fr_Flujo_Caja,Al_Cuota,Al_Intereses,Al_Capital_Amortizado,Al_Capital_Pendiente,Al_Flujo_Caja
0,6B4H6E,1,248.66,74.67,173.99,9826.01,-173.99,283.00,74.67,208.33,9791.67,-208.33
1,6B4H6E,2,248.66,73.37,175.29,9650.71,-175.29,281.44,73.11,208.33,9583.33,-208.33
2,6B4H6E,3,248.66,72.06,176.60,9474.11,-176.60,279.89,71.56,208.33,9375.00,-208.33
3,6B4H6E,4,248.66,70.74,177.92,9296.19,-177.92,278.33,70.00,208.33,9166.67,-208.33
4,6B4H6E,5,248.66,69.41,179.25,9116.94,-179.25,276.78,68.44,208.33,8958.33,-208.33
...,...,...,...,...,...,...,...,...,...,...,...,...
3795848,A8KWSO,20,1780.77,47.02,1733.74,7028.51,-1733.74,1711.39,44.72,1666.67,6666.67,-1666.67
3795849,A8KWSO,21,1780.77,37.72,1743.05,5285.46,-1743.05,1702.44,35.78,1666.67,5000.00,-1666.67
3795850,A8KWSO,22,1780.77,28.37,1752.40,3533.06,-1752.40,1693.50,26.83,1666.67,3333.33,-1666.67
3795851,A8KWSO,23,1780.77,18.96,1761.80,1771.26,-1761.80,1684.56,17.89,1666.67,1666.67,-1666.67
